# 09_02 Greedy and beam: is the likeliest next word the best translation?

A trained translator hands back a probability for every Spanish word at every step. This notebook loads
the lab's translator, looks inside one step, decodes the held-out sentences greedily and with a beam of
five, and scores both with chrF and BLEU. The BLEU you use, you finish writing yourself.

**How this notebook works.** Every notebook in this course has the same rhythm:

1. **Recall.** Answer from memory before you look anything up. `ask()` tells you at once whether you were right.
2. **Predict, then run.** Before a cell with a surprise in it, write your prediction into `guess()`. The next cell runs the code and `reveal()` compares.
3. **Worked example, then your turn.** One example is done in full; the next, near-identical one has lines marked `# YOUR CODE HERE`.
4. **Check.** A `check_...()` cell tests what you saved, exactly as the checkpoint will, and says what to fix.

Run cells in order with **Shift+Enter**. If you get lost, **Kernel, Restart Kernel and Run All Cells** starts clean.

In [ ]:
import json
import os
import time
import torch
import translate
from nlpcheck import ask, check_09_02, guess, reveal

torch.set_num_threads(4)
data = translate.load_split()
src, refs = translate.eval_split(data)
print(len(src), "held-out English sentences, with", sum(len(r) for r in refs), "Spanish references between them")

## 1. Recall

**r3.** During teacher-forced training, what is the decoder's input at step t? (a) the true word at step
t - 1, (b) its own guess at step t - 1, (c) the context vector again

**r4.** Roughly what is the loss of an untrained decoder over 8,004 Spanish entries? (a) 0.69, (b) 9.0, (c) 100

In [ ]:
ask("r3", "")
ask("r4", "")

## 2. The lab's translator

`translate.lab_model()` loads the model the session has been training in the background since it started:
`Seq2Seq(hidden=256)`, two passes over the 89,073 training pairs with Adam at a learning rate of 0.002,
seed 0. If the background job is still running, the cell says so and waits for it; it takes about ten
minutes of the session's time in all, so it is usually done by now. If it never ran, the cell trains the
model itself, which takes about as long, once.

In [ ]:
t = time.time()
model = translate.lab_model()
print(f"loaded in {time.time() - t:.0f} s:", sum(p.numel() for p in model.parameters()), "weights")
for i in (3, 250, 600, 900, 1100):
    print(" ".join(src[i]), "->", " ".join(translate.greedy(model, [src[i]], data)[0]))
    print("      reference:", " / ".join(" ".join(r) for r in refs[i][:2]))

## 3. Inside one step

The chapter describes each decoder output as a probability distribution over the whole vocabulary. Here is
the first step for "I am hungry.": the encoder's context, `<sos>` in, softmax out, and the five likeliest
first words.

In [ ]:
with torch.no_grad():
    h = model.encode(torch.tensor([translate.encode(translate.tokens("I am hungry."), data["src_itos"])]))
    logits, h = model.decode_step(torch.tensor([translate.SOS]), h)
    probs = torch.softmax(logits, -1)[0]
top = probs.topk(5)
for p, i in zip(top.values.tolist(), top.indices.tolist()):
    print(f"{data['tgt_itos'][i]:12} {p:.3f}")
print(f"the other {len(probs) - 5:,} entries share {1 - top.values.sum().item():.3f}")

Greedy decoding takes the top line and moves on; beam search keeps the top few and decides later.

## 4. Greedy against beam

Greedy decoding of all 1,174 sentences is one batched pass, a few seconds. Beam search with k = 5 decodes
each sentence on its own, keeping five prefixes, and takes up to about two and a half minutes. Predict: how many chrF points
will the beam add, rounded to a whole number? (0 means no difference; it can be negative.)

In [ ]:
guess("beam_gain", None)   # a whole number of chrF points

In [ ]:
t = time.time()
greedy_out = translate.greedy(model, src, data)
print(f"greedy: {time.time() - t:.0f} s")
t = time.time()
beam_out = [translate.beam(model, s, data, k=5) for s in src]
print(f"beam 5: {time.time() - t:.0f} s")
results = {"greedy_chrf": translate.chrf(greedy_out, refs), "beam_chrf": translate.chrf(beam_out, refs),
           "greedy_bleu": translate.bleu(greedy_out, refs), "beam_bleu": translate.bleu(beam_out, refs)}
for k, v in results.items():
    print(f"{k:12} {v:.2f}")
reveal("beam_gain", round(results["beam_chrf"] - results["greedy_chrf"]))
shown = 0
for s, g, b in zip(src, greedy_out, beam_out):
    if g != b and shown < 4:
        print(" ".join(s), "\n    greedy:", " ".join(g), "\n    beam:  ", " ".join(b))
        shown += 1

About two and a half points of chrF and three of BLEU, from the same weights and no training at all: in a
run measured for this lab, greedy 42.6 and beam 45.1 chrF, BLEU 27.0 and 30.3. Read the printed pairs
before you trust the average, though. Some are better and some are merely different: for "A fish can swim."
greedy wrote "un gato puede nadar" (a cat can swim) and the beam "nadar puede nadar" (swim can swim), which
is likelier under the model and no better. The beam finds sentences the model prefers; it cannot make the
model know that a fish is a "pez". It is not free either: it took about twenty times as long as greedy
decoding here, which is why production systems tune the width, and why a beam of 4 to 10 is common rather
than 50. Wider is not always better: very wide beams on models of this kind drift towards short, generic
sentences.

## 5. Your turn: finish BLEU

`my_bleu` below is `translate.bleu` with one line missing: the **clipped** match count. For each n-gram the
hypothesis contains, it earns matches up to the most times that n-gram appears in any **one** reference,
never more. `hc` counts the hypothesis's n-grams, `most` holds the highest count of each n-gram in any
reference. Write the line, run it on the greedy translations, and it must agree with `translate.bleu`.

In [ ]:
import collections
import math

def my_bleu(hyps, refs, max_n=4):
    match, total = [0] * max_n, [0] * max_n
    hyp_len = ref_len = 0
    for h, rs in zip(hyps, refs):
        hyp_len += len(h)
        ref_len += min((abs(len(r) - len(h)), len(r)) for r in rs)[1]    # the closest reference length
        for n in range(1, max_n + 1):
            hc = translate.ngrams(h, n)
            most = collections.Counter()
            for r in rs:
                most |= translate.ngrams(r, n)
            match[n - 1] += 0       # YOUR CODE HERE: replace 0 with the clipped matches, sum of min(count, most[g])
            total[n - 1] += max(len(h) - n + 1, 0)
    if min(match) == 0:
        return 0.0
    log_p = sum(math.log(m / t) for m, t in zip(match, total)) / max_n
    bp = 1.0 if hyp_len > ref_len else math.exp(1 - ref_len / max(hyp_len, 1))
    return 100 * bp * math.exp(log_p)

results["my_bleu"] = my_bleu(greedy_out, refs)
print(f"yours {results['my_bleu']:.3f}   translate.bleu {results['greedy_bleu']:.3f}")

In [ ]:
os.makedirs("out", exist_ok=True)
json.dump(results, open("out/09_02_results.json", "w"), indent=1)
check_09_02()

Without the clipping, a hypothesis of "la la la la" against a reference containing one "la" would score a
perfect unigram precision. The clip is the whole reason BLEU cannot be gamed by repetition, and the
brevity penalty is the reason it cannot be gamed by saying less.

## 6. Exit ticket

**x2.** Why does a perfect three-token translation get a BLEU of zero on its own? (a) BLEU ignores
punctuation, (b) it has no 4-grams, so the 4-gram precision is zero and so is the geometric mean, (c) the
brevity penalty

In [ ]:
ask("x2", "")

Explain it back: why is the likeliest first word not always the start of the likeliest sentence?

*Your explanation:* 